# Módulo 01: Características de la Vivienda y del Hogar (ENAHO 2024 - 2025)

Este notebook procesa y audita las características físicas del hábitat y servicios básicos de los hogares en **Lima Metropolitana y Callao** correspondientes a las encuestas ENAHO 2024 y 2025:
- **Filtro geográfico:** Ámbito metropolitano de Lima y Callao (códigos de departamento `07` y `15`).
- **Filtro de cobertura:** Entrevistas completas e incompletas con información suficiente (`RESULT` en `[1, 2]`).
- **Resolución de nulos estructurales:** Imputación intra-vivienda para hogares secundarios (`HOGAR > 11`) que comparten el predio físico con el hogar principal.
- **Ingeniería de Características:** Consolidación de tenencia de vivienda y registro SUNARP (`seguridad_tenencia`).
- **Salida estandarizada:** Exportación limpia y centralizada a `Data/processed/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Agregar raíz del proyecto a sys.path
ROOT_DIR = Path.cwd().resolve()
if 'notebooks' in str(ROOT_DIR):
    ROOT_DIR = ROOT_DIR.parents[1] if '01_modulos' in str(ROOT_DIR) else ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.pipeline import ENAHOPipeline
from src.processors.modulo01 import Modulo01Processor

## 1. Ejecución Automatizada del Pipeline Multi-Anual
Se procesan de manera secuencial y desacoplada las muestras de 2024 y 2025 usando `ENAHOPipeline`.

In [ ]:
pipeline = ENAHOPipeline(data_root=ROOT_DIR / "Data", verbose=True)

# Ejecutar procesamiento y exportación a Data/processed/
res_dict = pipeline.run_module("modulo01", years=[2024, 2025], export_cleaned=True, concatenate_years=False)
df_2024 = res_dict[2024]
df_2025 = res_dict[2025]

print(f"Dimensiones 2024: {df_2024.shape}")
print(f"Dimensiones 2025: {df_2025.shape}")

## 2. Auditoría de Calidad y Validación de Integridad (0% Nulos)

In [ ]:
for yr, df in [(2024, df_2024), (2025, df_2025)]:
    n_null = df.isnull().sum().sum()
    print(f"[Año {yr}] Nulos totales: {n_null} en {df.shape[0]} hogares evaluados")
    assert n_null == 0, f"Error: Se encontraron valores nulos en el dataset {yr}"
print("✓ Validación exitosa: 0% de valores nulos en todas las columnas para ambos años.")

## 3. Análisis Exploratorio de Datos (EDA) Multi-Anual

In [ ]:
df_all = pd.concat([df_2024, df_2025], axis=0)

# Tipo de Vivienda comparativo 2024 vs 2025
plt.figure(figsize=(10, 4))
piv_viv = pd.crosstab(df_all["tipo_vivienda"], df_all["anio_encuesta"], normalize="columns") * 100
ax = piv_viv.plot(kind="barh", figsize=(10, 4), colormap="viridis", edgecolor="black", alpha=0.85)
plt.title("Distribución Porcentual del Tipo de Vivienda (Lima y Callao)", fontsize=12, fontweight="bold")
plt.xlabel("Porcentaje de Hogares (%)")
plt.ylabel("Tipo de Vivienda")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Materiales de Vivienda: Paredes y Pisos
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

paredes_pct = df_all["material_paredes"].value_counts(normalize=True).mul(100).head(6)
paredes_pct.plot(kind="barh", ax=axes[0], color="#015D34", edgecolor="black")
axes[0].set_title("Material Predominante en Paredes (%)", fontweight="bold")
axes[0].set_xlabel("Porcentaje (%)")
axes[0].grid(axis="x", linestyle="--", alpha=0.5)

pisos_pct = df_all["material_pisos"].value_counts(normalize=True).mul(100).head(6)
pisos_pct.plot(kind="barh", ax=axes[1], color="#009A74", edgecolor="black")
axes[1].set_title("Material Predominante en Pisos (%)", fontweight="bold")
axes[1].set_xlabel("Porcentaje (%)")
axes[1].grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Servicios Básicos y Seguridad de Tenencia
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

df_all["seguridad_tenencia"].value_counts().plot(kind="bar", ax=axes[0], color="#40B497", edgecolor="black")
axes[0].set_title("Seguridad de Tenencia y Registro", fontweight="bold")
axes[0].set_ylabel("Cantidad de Hogares")
axes[0].tick_params(axis='x', rotation=30)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

df_all["total_habitaciones"].plot(kind="hist", bins=15, ax=axes[1], color="#015D34", edgecolor="black")
axes[1].set_title("Distribución de Habitaciones por Hogar", fontweight="bold")
axes[1].set_xlabel("Número de Habitaciones")
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()